# Q1

In [ ]:
suits = ['Hearts', 'Diamonds', 'Spades', 'Clubs']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'Jack', 'Queen', 'King', 'Ace']

deck = []
for suit in suits:
    for rank in ranks:
        deck.append((rank, suit))

red_cards = []
for rank, suit in deck:
    if suit == 'Hearts' or suit == 'Diamonds':
        red_cards.append((rank, suit))

face_cards = []
for rank, suit in deck:
    if rank == 'Jack' or rank == 'Queen' or rank == 'King':
        face_cards.append((rank, suit))

# Q1: P(Red)
print(f"Q1: {len(red_cards)}/{len(deck)} = {len(red_cards)/len(deck):.4f}")

# Q2: P(Heart | Red)
hearts_in_red = []
for rank, suit in red_cards:
    if suit == 'Hearts':
        hearts_in_red.append((rank, suit))
print(f"Q2: {len(hearts_in_red)}/{len(red_cards)} = {len(hearts_in_red)/len(red_cards):.4f}")

# Q3: P(Diamond | Face)
diamonds_in_face = []
for rank, suit in face_cards:
    if suit == 'Diamonds':
        diamonds_in_face.append((rank, suit))
print(f"Q3: {len(diamonds_in_face)}/{len(face_cards)} = {len(diamonds_in_face)/len(face_cards):.4f}")

# Q4: P(Spade or Queen | Face)
spade_or_queen_in_face = []
for rank, suit in face_cards:
    if suit == 'Spades' or rank == 'Queen':
        spade_or_queen_in_face.append((rank, suit))
print(f"Q4: {len(spade_or_queen_in_face)}/{len(face_cards)} = {len(spade_or_queen_in_face)/len(face_cards):.4f}")

Q1: 26/52 = 0.5000
Q2: 13/26 = 0.5000
Q3: 3/12 = 0.2500
Q4: 6/12 = 0.5000


# Q2

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('StudyHours', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_intelligence = TabularCPD(
    variable = 'Intelligence',
    variable_card = 2,
    values = [[0.3],[0.7]]
)

cpd_StudyHours = TabularCPD(
    variable = 'StudyHours',
    variable_card = 2,
    values = [[0.4],[0.6]]
)

cpd_diff = TabularCPD(
    variable = 'Difficulty',
    variable_card = 2,
    values = [[0.6], [0.4]]
)

cpd_grade = TabularCPD(
    variable = 'Grade',
    variable_card = 3,
    values = [
        [0.2, 0.05, 0.4, 0.1, 0.7, 0.3, 0.9, 0.5],
        [0.3, 0.25, 0.4, 0.3, 0.2, 0.4, 0.08, 0.4],
        [0.5, 0.7, 0.2, 0.6, 0.1, 0.3, 0.02, 0.1]
    ],
    evidence = ['Intelligence', 'StudyHours', 'Difficulty'],
    evidence_card = [2, 2, 2]
)

cpd_pass = TabularCPD(
    variable = 'Pass',
    variable_card = 2,
    values = [
        [0.05, 0.2, 0.5],
        [0.95, 0.80, 0.5]
    ],
    evidence = ['Grade'],
    evidence_card = [3]
)

model.add_cpds(cpd_intelligence, cpd_StudyHours, cpd_diff, cpd_grade, cpd_pass
               )

assert model.check_model(), 'Model is incorrect'

inference = VariableElimination(model)
result = inference.query(
    variables = ['Pass'],
    evidence = {
        'StudyHours' : 1,
        'Difficulty' : 1 
    }
)

print(result)


res = inference.query(
    variables = ['Intelligence'],
    evidence = {
        'Pass' : 1
    }
)

print(res)

+---------+-------------+
| Pass    |   phi(Pass) |
+=========+=============+
| Pass(0) |      0.2180 |
+---------+-------------+
| Pass(1) |      0.7820 |
+---------+-------------+
+-----------------+---------------------+
| Intelligence    |   phi(Intelligence) |
+=================+=====================+
| Intelligence(0) |              0.2566 |
+-----------------+---------------------+
| Intelligence(1) |              0.7434 |
+-----------------+---------------------+


# Q3

In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'),
    ('Disease', 'Cough'),
    ('Disease', 'Fatigue'),
    ('Disease', 'Chills'),
])

cpd_disease = TabularCPD(
    variable='Disease',
    variable_card=2,
    values=[[0.3], [0.7]]
)

cpd_fever = TabularCPD(
    variable='Fever',
    variable_card=2,
    values=[[0.1, 0.5], [0.9, 0.5]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_cough = TabularCPD(
    variable='Cough',
    variable_card=2,
    values=[[0.2, 0.4], [0.8, 0.6]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_fatigue = TabularCPD(
    variable='Fatigue',
    variable_card=2,
    values=[[0.3, 0.7], [0.7, 0.3]],
    evidence=['Disease'],
    evidence_card=[2]
)

cpd_chills = TabularCPD(
    variable='Chills',
    variable_card=2,
    values=[[0.4, 0.6], [0.6, 0.4]],
    evidence=['Disease'],
    evidence_card=[2]
)

model.add_cpds(cpd_disease, cpd_cough, cpd_chills, cpd_fatigue, cpd_fever)

assert model.check_model(), 'Model is incorrect'

inference = VariableElimination(model)

In [8]:
res = inference.query(
    variables=['Disease'],
    evidence={
        'Fever': 1,
        'Cough': 1
    }
)

print(res)

+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.5070 |
+------------+----------------+
| Disease(1) |         0.4930 |
+------------+----------------+


In [10]:
res = inference.query(
    variables=['Disease'],
    evidence={
        'Fever': 1,
        'Cough': 1,
        'Chills': 1
    }
)

print(res)

+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.6067 |
+------------+----------------+
| Disease(1) |         0.3933 |
+------------+----------------+


In [12]:
res = inference.query(
    variables=['Fatigue'],
    evidence={
        'Disease': 0
    }
)

print(res)

+------------+----------------+
| Fatigue    |   phi(Fatigue) |
+============+================+
| Fatigue(0) |         0.3000 |
+------------+----------------+
| Fatigue(1) |         0.7000 |
+------------+----------------+


# Q4

In [16]:
import numpy as np

states = ["Sunny", "Cloudy", "Rainy"]
transition_matrix = [
    [0.89, 0.08, 0.03],     # sunny
    [0.1, 0.7, 0.2],     # cloudy
    [0.02, 0.4, 0.58]      # rainy
]


def simulate(init_state, iterations):
    current_state = init_state
    state_seq = [current_state]

    for _ in range(iterations):
        if current_state == "Sunny":
            next_state = np.random.choice(states, p = transition_matrix[0])
        elif current_state == "Cloudy":
            next_state = np.random.choice(states, p = transition_matrix[1])
        else:
            next_state = np.random.choice(states, p = transition_matrix[2])
        
        state_seq.append(next_state)
        current_state = next_state
    return state_seq

init = "Sunny"
steps = 10
n = 1000
at_least_3_rainy = 0

for _ in range(n):
    seq = simulate(init, steps)

    rainy_days = seq[1:].count("Rainy")
    if rainy_days >= 3:
        at_least_3_rainy += 1

probability = at_least_3_rainy / n
print(f"Probability of at least 3 rainy days: {probability:.4f}")

Probability of at least 3 rainy days: 0.2010
